# ⚡ Docker Optimization

**Make containers faster and smaller**

## 📋 Overview

**What you'll learn:**
- Image size optimization
- Build caching
- Layer optimization
- Performance tuning

**Time estimate:** ⏱️ 45 minutes

## 📦 Image Size Optimization

### Before (1.2 GB):
```dockerfile
FROM python:3.11
COPY . .
RUN pip install -r requirements.txt
```

### After (300 MB):
```dockerfile
# Use slim base
FROM python:3.11-slim

# Multi-stage build
FROM python:3.11-slim AS builder
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

FROM python:3.11-slim
COPY --from=builder /root/.local /root/.local
COPY . .

# Result: 300 MB (75% smaller!)
```

## 🚀 Build Caching

### Bad (cache misses):
```dockerfile
COPY . .
RUN pip install -r requirements.txt  # Rebuilds every time
```

### Good (cache hits):
```dockerfile
# Copy only requirements first
COPY requirements.txt .
RUN pip install -r requirements.txt  # Cached if requirements unchanged

# Copy code last (changes frequently)
COPY . .
```

**Build times:**
```
Bad:  2 minutes every build
Good: 2 minutes first, then 10 seconds
```

## 🔧 Layer Optimization

### Combine commands:
```dockerfile
# ❌ Bad - 3 layers
RUN apt-get update
RUN apt-get install -y curl
RUN rm -rf /var/lib/apt/lists/*

# ✅ Good - 1 layer
RUN apt-get update && \
    apt-get install -y curl && \
    rm -rf /var/lib/apt/lists/*
```

### Remove temporary files:
```dockerfile
# ❌ Bad - temp files in layer
RUN wget https://example.com/file.tar.gz && \
    tar -xzf file.tar.gz

# ✅ Good - cleanup in same layer
RUN wget https://example.com/file.tar.gz && \
    tar -xzf file.tar.gz && \
    rm file.tar.gz
```

## ⚡ Performance Tuning

### Use BuildKit:
```bash
# Enable BuildKit (faster builds)
DOCKER_BUILDKIT=1 docker build -t app .
```

### Parallel builds:
```dockerfile
# Dockerfile supports parallel stages
FROM base AS stage1
...

FROM base AS stage2
...

FROM base AS final
COPY --from=stage1 ...
COPY --from=stage2 ...
```

### Resource limits:
```yaml
# docker-compose.yml
services:
  api:
    deploy:
      resources:
        limits:
          cpus: '2'
          memory: 4G
        reservations:
          cpus: '1'
          memory: 2G
```

## ✅ Summary

**Optimization checklist:**

1. **Image size**
   - Use slim/alpine base images
   - Multi-stage builds
   - Remove unnecessary files

2. **Build speed**
   - Order layers correctly
   - Use BuildKit
   - Cache dependencies

3. **Runtime performance**
   - Set resource limits
   - Use production WSGI server
   - Enable health checks

**Results:**
```
Image size:  1.2 GB → 300 MB (75% reduction)
Build time:  2 min → 10 sec (92% faster)
Boot time:   15 sec → 3 sec (80% faster)
```

### Next: `13_docker/05_production.ipynb`